# Notebook 2 — Original Dangling Centrality on a BioGPT Attention Graph

This notebook loads the graph exported by Notebook 1 and calculates the original **Dangling Centrality** \(\phi_C\):

\[
\phi_C(G)=\sum_{i<j,\;d_{ij}>0}\frac{1}{d_{ij}}
\]

\[
\phi_C(v)=
\frac{\phi_C(G)-\phi_C(G_v^{d})}
{\phi_C(G)}
\]

where \(G_v^{d}\) is obtained by retaining node \(v\) but removing all of its incident links.

The original formulation evaluates an undirected graph and divides the reciprocal-distance sum by two because each unordered pair appears twice in the symmetric shortest-path distance matrix. Computationally, summing only \(i<j\) is exactly equivalent.

## Attention-graph treatment

Notebook 1 produces a directed causal attention graph. For the **primary original Dangling Centrality analysis**, this notebook constructs an undirected structural projection:

- every retained attention relationship becomes an undirected communication link;
- reciprocal links are merged;
- shortest-path lengths are measured in hops;
- attention weights are preserved as node/edge metadata but do not alter the original SPDM definition.

This keeps the centrality formula unchanged. A weighted extension should be reported separately and must not be presented as the original original metric.

## 1. Install and import packages

In [ ]:
!pip -q install networkx openpyxl scipy

In [ ]:
import json
import math
import pickle
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, pearsonr, spearmanr

warnings.filterwarnings("ignore")

print("NetworkX:", nx.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Upload or locate Notebook 1 outputs

In [ ]:
# In Google Colab, run this cell and upload notebook1_outputs.zip.
# If the file is already present at /content/notebook1_outputs.zip,
# no upload is required.

ZIP_PATH = Path("/content/notebook1_outputs.zip")
EXTRACT_DIR = Path("/content/notebook1_outputs")

if not ZIP_PATH.exists() and not EXTRACT_DIR.exists():
    from google.colab import files
    uploaded = files.upload()

    zip_candidates = [
        Path("/content") / filename
        for filename in uploaded
        if filename.lower().endswith(".zip")
    ]

    if not zip_candidates:
        raise FileNotFoundError(
            "Upload the notebook1_outputs.zip archive created by Notebook 1."
        )

    ZIP_PATH = zip_candidates[0]

if ZIP_PATH.exists():
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_DIR))

print("Input directory:", EXTRACT_DIR)
print("Files:")
for path in sorted(EXTRACT_DIR.iterdir()):
    print(" -", path.name)

## 3. Load the attention graph and metadata

In [ ]:
GRAPH_PATH = EXTRACT_DIR / "attention_graph.gpickle"
METADATA_PATH = EXTRACT_DIR / "metadata.json"

if not GRAPH_PATH.exists():
    raise FileNotFoundError(f"Missing graph file: {GRAPH_PATH}")

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Missing metadata file: {METADATA_PATH}")

with open(GRAPH_PATH, "rb") as file:
    directed_attention_graph = pickle.load(file)

with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

if not isinstance(directed_attention_graph, nx.DiGraph):
    raise TypeError(
        "Notebook 1 should provide a directed NetworkX attention graph."
    )

print("Question ID:", metadata.get("question_id"))
print("Question:", metadata.get("question"))
print("Correct answer:", metadata.get("correct_option_letter"),
      "-", metadata.get("correct_answer"))
print("Directed graph nodes:", directed_attention_graph.number_of_nodes())
print("Directed graph edges:", directed_attention_graph.number_of_edges())

## 4. Construct the undirected structural projection

In [ ]:
def make_original_structural_projection(directed_graph):
    '''
    Construct an undirected simple graph for the original SPDM formulation.

    If one or both directed edges exist between a node pair, one undirected
    communication link is retained. The maximum attention value is stored only
    as metadata; original Dangling Centrality uses unweighted hop distance.
    '''
    graph = nx.Graph()

    for node_id, data in directed_graph.nodes(data=True):
        graph.add_node(node_id, **data)

    pair_weights = {}

    for source, target, data in directed_graph.edges(data=True):
        if source == target:
            continue

        pair = tuple(sorted((source, target)))
        weight = float(data.get("weight", data.get("attention", 1.0)))
        pair_weights[pair] = max(pair_weights.get(pair, 0.0), weight)

    for (source, target), weight in pair_weights.items():
        graph.add_edge(
            source,
            target,
            attention_weight=weight,
            weight=weight
        )

    return graph


G = make_original_structural_projection(directed_attention_graph)

print("Original-analysis graph type:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Undirected edges:", G.number_of_edges())
print("Density:", nx.density(G))
print("Connected components:", nx.number_connected_components(G))

## 5. Original communication score from the SPDM

In [ ]:
def reciprocal_distance_communication(graph):
    '''
    Original network communication score.

    For an undirected graph, sum 1/d(i,j) over all connected unordered
    node pairs i < j. Disconnected pairs and diagonal entries contribute zero.

    This is equivalent to summing the element-wise inverse of the symmetric
    SPDM and dividing the total by two.
    '''
    nodes = list(graph.nodes())
    node_order = {node: index for index, node in enumerate(nodes)}
    communication = 0.0

    all_lengths = dict(nx.all_pairs_shortest_path_length(graph))

    for source in nodes:
        source_index = node_order[source]

        for target, distance in all_lengths[source].items():
            target_index = node_order[target]

            if target_index <= source_index:
                continue

            if distance > 0:
                communication += 1.0 / float(distance)

    return communication


original_communication = reciprocal_distance_communication(G)

if original_communication <= 0:
    raise RuntimeError("The original graph communication score is zero.")

print("Original communication score phi_C(G):",
      round(original_communication, 8))

## 6. Validate the implementation on the five-node example from the publication

In [ ]:
# Publication example:
# edges = (0,1), (0,2), (0,3), (1,2), (3,4)
# The reported original communication is approximately 7.167.
# After dangling node 0, communication is 2.
# Therefore phi_C(0) is approximately 0.7209.

validation_graph = nx.Graph()
validation_graph.add_edges_from([
    (0, 1),
    (0, 2),
    (0, 3),
    (1, 2),
    (3, 4)
])

validation_original = reciprocal_distance_communication(validation_graph)

validation_dangling = validation_graph.copy()
validation_dangling.remove_edges_from(
    list(validation_dangling.edges(0))
)
validation_after = reciprocal_distance_communication(validation_dangling)

validation_phi = (
    validation_original - validation_after
) / validation_original

print("Validation original communication:", validation_original)
print("Validation after dangling node 0:", validation_after)
print("Validation phi_C(0):", validation_phi)

assert np.isclose(validation_original, 7.166666666666667)
assert np.isclose(validation_after, 2.0)
assert np.isclose(validation_phi, 0.7209302325581395)

print("Original-formula validation passed.")

## 7. Calculate Dangling Centrality for every token node

In [ ]:
def dangling_centrality_original(graph):
    '''
    Compute original Dangling Centrality by removing every incident edge of
    one node at a time while retaining the isolated node.
    '''
    baseline = reciprocal_distance_communication(graph)

    if baseline <= 0:
        raise ValueError("Baseline graph communication must be positive.")

    results = []
    start_time = time.time()

    for iteration, node_id in enumerate(graph.nodes(), start=1):
        modified_graph = graph.copy()
        incident_edges = list(modified_graph.edges(node_id))
        modified_graph.remove_edges_from(incident_edges)

        modified_communication = reciprocal_distance_communication(
            modified_graph
        )

        phi_c = (
            baseline - modified_communication
        ) / baseline

        results.append({
            "node_id": node_id,
            "token": str(graph.nodes[node_id].get("token", node_id)),
            "position": int(graph.nodes[node_id].get("position", node_id)),
            "removed_incident_edges": len(incident_edges),
            "original_communication": baseline,
            "dangling_communication": modified_communication,
            "communication_loss": baseline - modified_communication,
            "dangling_centrality_phi_c": phi_c
        })

        if iteration % 10 == 0 or iteration == graph.number_of_nodes():
            print(
                f"Processed {iteration}/{graph.number_of_nodes()} nodes"
            )

    elapsed = time.time() - start_time

    ranking = pd.DataFrame(results).sort_values(
        ["dangling_centrality_phi_c", "position"],
        ascending=[False, True]
    ).reset_index(drop=True)

    ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))

    return ranking, elapsed


dangling_ranking, dangling_elapsed_seconds = (
    dangling_centrality_original(G)
)

print("Computation time:", round(dangling_elapsed_seconds, 3), "seconds")
display(dangling_ranking.head(20))

## 8. Add conventional centralities for comparison

In [ ]:
degree = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G, normalized=True)
closeness = nx.closeness_centrality(G)

try:
    eigenvector = nx.eigenvector_centrality(
        G,
        max_iter=5000,
        tol=1e-10
    )
except nx.PowerIterationFailedConvergence:
    eigenvector = {
        node: np.nan
        for node in G.nodes()
    }

comparison_table = dangling_ranking.copy()

comparison_table["degree_centrality"] = comparison_table["node_id"].map(degree)
comparison_table["betweenness_centrality"] = (
    comparison_table["node_id"].map(betweenness)
)
comparison_table["closeness_centrality"] = (
    comparison_table["node_id"].map(closeness)
)
comparison_table["eigenvector_centrality"] = (
    comparison_table["node_id"].map(eigenvector)
)

display(comparison_table.head(20))

## 9. Correlation analysis

In [ ]:
def safe_correlations(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    x_valid = np.asarray(x)[valid]
    y_valid = np.asarray(y)[valid]

    if len(x_valid) < 3:
        return {
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "spearman_rho": np.nan,
            "spearman_p": np.nan,
            "kendall_tau": np.nan,
            "kendall_p": np.nan
        }

    pearson = pearsonr(x_valid, y_valid)
    spearman = spearmanr(x_valid, y_valid)
    kendall = kendalltau(x_valid, y_valid)

    return {
        "pearson_r": pearson.statistic,
        "pearson_p": pearson.pvalue,
        "spearman_rho": spearman.statistic,
        "spearman_p": spearman.pvalue,
        "kendall_tau": kendall.statistic,
        "kendall_p": kendall.pvalue
    }


centrality_columns = {
    "Degree Centrality": "degree_centrality",
    "Betweenness Centrality": "betweenness_centrality",
    "Closeness Centrality": "closeness_centrality",
    "Eigenvector Centrality": "eigenvector_centrality"
}

correlation_rows = []

phi_values = comparison_table["dangling_centrality_phi_c"].to_numpy()

for measure_name, column_name in centrality_columns.items():
    correlations = safe_correlations(
        phi_values,
        comparison_table[column_name].to_numpy()
    )

    correlation_rows.append({
        "comparison_measure": measure_name,
        **correlations
    })

correlation_table = pd.DataFrame(correlation_rows)
display(correlation_table)

## 10. Create IEEE-ready result tables

In [ ]:
TOP_N = 15

ieee_top_nodes = comparison_table.head(TOP_N).copy()
ieee_top_nodes = ieee_top_nodes[
    [
        "rank",
        "node_id",
        "token",
        "position",
        "removed_incident_edges",
        "dangling_centrality_phi_c",
        "degree_centrality",
        "betweenness_centrality",
        "closeness_centrality"
    ]
]

numeric_columns = ieee_top_nodes.select_dtypes(
    include=[np.number]
).columns

ieee_top_nodes[numeric_columns] = (
    ieee_top_nodes[numeric_columns].round(6)
)

graph_summary = pd.DataFrame([
    {
        "question_id": metadata.get("question_id"),
        "question": metadata.get("question"),
        "correct_option": metadata.get("correct_option_letter"),
        "correct_answer": metadata.get("correct_answer"),
        "directed_attention_nodes": directed_attention_graph.number_of_nodes(),
        "directed_attention_edges": directed_attention_graph.number_of_edges(),
        "undirected_projection_nodes": G.number_of_nodes(),
        "undirected_projection_edges": G.number_of_edges(),
        "undirected_density": nx.density(G),
        "connected_components": nx.number_connected_components(G),
        "original_communication_phi_c_G": original_communication,
        "maximum_node_phi_c": dangling_ranking[
            "dangling_centrality_phi_c"
        ].max(),
        "mean_node_phi_c": dangling_ranking[
            "dangling_centrality_phi_c"
        ].mean(),
        "median_node_phi_c": dangling_ranking[
            "dangling_centrality_phi_c"
        ].median(),
        "dangling_computation_seconds": dangling_elapsed_seconds
    }
])

display(ieee_top_nodes)
display(graph_summary.T.rename(columns={0: "value"}))

## 11. Publication-quality top-node bar chart

In [ ]:
top_plot = dangling_ranking.head(15).sort_values(
    "dangling_centrality_phi_c",
    ascending=True
)

plot_labels = [
    f"{row.token} [{row.node_id}]"
    for row in top_plot.itertuples()
]

plt.figure(figsize=(10, 7))
plt.barh(
    plot_labels,
    top_plot["dangling_centrality_phi_c"]
)
plt.xlabel(r"Original Dangling Centrality, $\phi_C$")
plt.ylabel("Token node")
plt.title("Most Structurally Influential BioGPT Attention Tokens")
plt.tight_layout()

bar_chart_path = EXTRACT_DIR / "dangling_centrality_top15.png"
plt.savefig(bar_chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", bar_chart_path)

## 12. Highlight the most influential nodes in the graph

In [ ]:
TOP_HIGHLIGHT = 10
top_node_ids = set(
    dangling_ranking.head(TOP_HIGHLIGHT)["node_id"].tolist()
)

layout = nx.spring_layout(G, seed=42, k=1.25)

node_sizes = [
    1200 if node_id in top_node_ids else 300
    for node_id in G.nodes()
]

node_widths = [
    2.5 if node_id in top_node_ids else 0.5
    for node_id in G.nodes()
]

labels = {
    node_id: G.nodes[node_id].get("token", str(node_id))
    for node_id in G.nodes()
}

plt.figure(figsize=(18, 14))

nx.draw_networkx_edges(
    G,
    layout,
    alpha=0.25,
    width=0.8
)

nx.draw_networkx_nodes(
    G,
    layout,
    node_size=node_sizes,
    linewidths=node_widths,
    edgecolors="black",
    alpha=0.9
)

nx.draw_networkx_labels(
    G,
    layout,
    labels=labels,
    font_size=8
)

plt.title(
    "BioGPT Attention Graph with Top Dangling-Centrality Tokens Highlighted"
)
plt.axis("off")
plt.tight_layout()

highlight_path = EXTRACT_DIR / "dangling_centrality_highlighted_graph.png"
plt.savefig(highlight_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", highlight_path)

## 13. Export CSV and Excel tables

In [ ]:
dangling_ranking.to_csv(
    EXTRACT_DIR / "dangling_centrality_full_ranking.csv",
    index=False
)

comparison_table.to_csv(
    EXTRACT_DIR / "centrality_comparison_full.csv",
    index=False
)

correlation_table.to_csv(
    EXTRACT_DIR / "centrality_correlations.csv",
    index=False
)

ieee_top_nodes.to_csv(
    EXTRACT_DIR / "ieee_top_dangling_nodes.csv",
    index=False
)

graph_summary.to_csv(
    EXTRACT_DIR / "graph_and_dangling_summary.csv",
    index=False
)

excel_path = EXTRACT_DIR / "notebook2_dangling_centrality_tables.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    graph_summary.to_excel(
        writer,
        sheet_name="Graph Summary",
        index=False
    )
    ieee_top_nodes.to_excel(
        writer,
        sheet_name="IEEE Top Nodes",
        index=False
    )
    dangling_ranking.to_excel(
        writer,
        sheet_name="Full PhiC Ranking",
        index=False
    )
    comparison_table.to_excel(
        writer,
        sheet_name="Centrality Comparison",
        index=False
    )
    correlation_table.to_excel(
        writer,
        sheet_name="Correlations",
        index=False
    )

print("Excel workbook:", excel_path)

## 14. Create Notebook 2 output archive

In [ ]:
NOTEBOOK2_OUTPUT_DIR = Path("/content/notebook2_outputs")
NOTEBOOK2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_names = [
    "dangling_centrality_full_ranking.csv",
    "centrality_comparison_full.csv",
    "centrality_correlations.csv",
    "ieee_top_dangling_nodes.csv",
    "graph_and_dangling_summary.csv",
    "notebook2_dangling_centrality_tables.xlsx",
    "dangling_centrality_top15.png",
    "dangling_centrality_highlighted_graph.png"
]

for filename in output_names:
    source = EXTRACT_DIR / filename
    if not source.exists():
        raise FileNotFoundError(f"Expected output missing: {source}")
    shutil.copy2(source, NOTEBOOK2_OUTPUT_DIR / filename)

methodology = {
    "primary_metric": "Original Dangling Centrality",
    "publication": (
        "Fatima, Hina, and Wasif, Scientific Reports 15, 41078 (2025)"
    ),
    "doi": "10.1038/s41598-025-24930-8",
    "graph_treatment": (
        "Undirected unweighted structural projection of the directed "
        "BioGPT attention graph"
    ),
    "baseline_communication": original_communication,
    "formula": (
        "phi_C(v) = [phi_C(G) - phi_C(G_v^d)] / phi_C(G)"
    ),
    "node_operation": (
        "Retain node v and remove every incident edge"
    ),
    "disconnected_pairs": "Excluded from reciprocal-distance sum",
    "self_distances": "Excluded",
    "unordered_pair_counting": (
        "Sum over i<j, equivalent to symmetric SPDM sum divided by 2"
    )
}

with open(
    NOTEBOOK2_OUTPUT_DIR / "methodology.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(methodology, file, indent=2)

archive_path = shutil.make_archive(
    "/content/notebook2_outputs",
    "zip",
    root_dir=NOTEBOOK2_OUTPUT_DIR
)

print("Notebook 2 archive:", archive_path)

# To download automatically in Colab, uncomment:
# from google.colab import files
# files.download(archive_path)

## 15. Final validation and main result

In [ ]:
required_output_files = output_names + ["methodology.json"]

missing_output_files = [
    filename
    for filename in required_output_files
    if not (NOTEBOOK2_OUTPUT_DIR / filename).exists()
]

if missing_output_files:
    raise RuntimeError(
        f"Missing Notebook 2 outputs: {missing_output_files}"
    )

top_result = dangling_ranking.iloc[0]

print("=" * 76)
print("NOTEBOOK 2 COMPLETED SUCCESSFULLY")
print("=" * 76)
print("Original baseline communication:", original_communication)
print("Highest-ranked token:", top_result["token"])
print("Node ID:", top_result["node_id"])
print("Original Dangling Centrality:",
      top_result["dangling_centrality_phi_c"])
print("Nodes evaluated:", len(dangling_ranking))
print("Excel tables:", excel_path)
print("ZIP archive:", archive_path)
print("=" * 76)

display(ieee_top_nodes)